There are a few different ways to check how well we sample a timescale in MAF. Let's compare `TgapsPercentMetric` and `GapsMetric`

In [ ]:
import numpy as np
from rubin_sim.maf import GapsMetric, TgapsPercentMetric
import matplotlib.pylab as plt

%matplotlib inline

In [ ]:
# get the dtype as expected
data = np.zeros(10, dtype=[("observationStartMJD", float)])

# Define a set of times
data["observationStartMJD"] += np.arange(10)

In [ ]:
data

In [ ]:
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.25) - 0.125)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

In [ ]:
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.5))
plt.xlabel("time (days)")
plt.ylabel("N obs")

In [ ]:
# TgapsPercent will tell us what percentage of consecutive observations are in the range 0.5 to 1 day
tgp = TgapsPercentMetric(min_time=0.5, max_time=1.0)

# Gaps will tell us how many times we have independently sampled the 0.5-1.5 day timescale.
gaps = GapsMetric(time_scale=24.0)

In [ ]:
tgp.run(data)

In [ ]:
gaps.run(data)

In [ ]:
# what this is telling us--we are 100% optimized for observing at the 1-day timescale.
# And we have observed the timescale of interest 8 times

In [ ]:
# what happens if we double the number of observations by increasing the frequency
data = np.zeros(20, dtype=[("observationStartMJD", float)])

# Define a set of times
data["observationStartMJD"] += np.arange(0, 10, 0.5)

In [ ]:
data

In [ ]:
tgp.run(data)

In [ ]:
gaps.run(data)

In [ ]:
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.25) - 0.125)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(
    data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25, edgecolor="black", linewidth=1.2
)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

In [ ]:
# and increasing the frequency again
# what happens if we double the number of observations
data = np.zeros(40, dtype=[("observationStartMJD", float)])

# Define a set of times
data["observationStartMJD"] += np.arange(0, 10, 0.25)

In [ ]:
tgp.run(data)

In [ ]:
gaps.run(data)

In [ ]:
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.1) - 0.05)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(
    data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25, edgecolor="black", linewidth=1.2
)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

This shows one issue with `TgapsPercent`--if we have very high sampling, it (correctly) reports that we are very un-optimized in terms of observing the timescale. This is not how science metrics typically behave. Normally, we want our metrics to only increase (or stay constant) as more observations are added. This is what the `Gaps` metric does. 

In [ ]:
# Let's do 10 days at 1-day cadence, 10 days at 0.5-day cadence

# what happens if we double the number of observations
# get the dtype as expected
data = np.zeros(30, dtype=[("observationStartMJD", float)])

# Define a set of times
data["observationStartMJD"] += np.concatenate((np.arange(0, 10, 1), np.arange(10, 20, 0.5)))

In [ ]:
data

In [ ]:
tgp.run(data)

In [ ]:
gaps.run(data)

In [ ]:
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.1) - 0.05)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(
    data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25, edgecolor="black", linewidth=1.2
)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

We once again have a 100% from `TgapsPercent`, so we might think this sequence is just as good as out initial sequence of 10 days. But `Gaps` correctly notes that now we have more days independently sampled, so it increases.

In [ ]:
# Let's do 10 days at 1-day cadence, 10 days at 0.25-day cadence

# what happens if we double the number of observations
# get the dtype as expected
data = np.zeros(50, dtype=[("observationStartMJD", float)])

# Define a set of times
data["observationStartMJD"] += np.concatenate((np.arange(0, 10, 1), np.arange(10, 20, 0.25)))

In [ ]:
data

In [ ]:
tgp.run(data)

In [ ]:
gaps.run(data)

In [ ]:
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.1) - 0.05)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(
    data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25, edgecolor="black", linewidth=1.2
)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

Once again, adding data has caused `TgapsPercent` to drop, while `Gaps` stays nearly constant.